# Representation Similarity

Analyze cosine similarity between the final prompt-token representations of edit, paraphrase, true-answer locality, and new-answer locality prompts. The notebook filters prompts by top-1 next-token correctness, saves record-level results to `data/similarity/{model}.json`, and plots aggregate mean with one-standard-deviation bands.

## Environment And Configuration

In [ ]:
from __future__ import annotations

import gc
import json
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from random import Random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_KEY = "qwen2.5-7b"
# MODEL_KEY = "qwen3.5-9b"
# MODEL_KEY = "llama3.1-8b"

BASE_CONFIG_PATH = PROJECT_ROOT / "src" / "config" / "base" / f"{MODEL_KEY}.yaml"
with open(BASE_CONFIG_PATH, encoding="utf-8") as file:
    base_config = yaml.safe_load(file)

DATASET_REPO = "Savassan/KnowledgeSurgeon"
DATASET_CONFIG = "es_loc"
DATASET_SPLIT = "data"
OUTPUT_DIR = PROJECT_ROOT / "data" / "similarity"


def similarity_output_path(record_limit: int | None) -> Path:
    if record_limit is not None and int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    output_stem = MODEL_KEY if record_limit is None else f"{MODEL_KEY}.limit-{int(record_limit)}"
    return OUTPUT_DIR / f"{output_stem}.json"


print(f"model={MODEL_KEY} checkpoint={base_config['model_name']}")
print(f"output_dir={OUTPUT_DIR}")

## Load Dataset And Base Model

In [ ]:
dataset = None
model = None
tokenizer = None
backbone = None
output_head = None
decoder_layers = None
input_device = None
output_device = None
model_source = base_config["model_name"]


def load_similarity_resources() -> None:
    global dataset, model, tokenizer, backbone, output_head
    global decoder_layers, input_device, output_device

    if model is not None:
        return

    dataset = load_dataset(DATASET_REPO, DATASET_CONFIG, split=DATASET_SPLIT)
    model_parallel = bool(base_config.get("model_parallel", False))
    model_dtype = torch.bfloat16 if base_config.get("fp16", True) else torch.float32
    model_kwargs = {"dtype": model_dtype}
    if model_parallel:
        model_kwargs["device_map"] = base_config.get("device_map") or "auto"

    model = AutoModelForCausalLM.from_pretrained(model_source, **model_kwargs)
    if not model_parallel:
        device = torch.device(
            f"cuda:{int(base_config.get('gpu_id', 0))}"
            if torch.cuda.is_available()
            else "cpu"
        )
        model = model.to(device)
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(model_source)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_model_prefix = getattr(model, "base_model_prefix", None)
    backbone = getattr(model, base_model_prefix, None) if base_model_prefix else None
    if backbone is None or backbone is model:
        backbone = model.base_model
    if backbone is model:
        raise RuntimeError("Could not resolve the causal language model backbone.")

    output_head = model.get_output_embeddings()
    decoder_layers = getattr(backbone, "layers", None)
    if decoder_layers is None:
        raise RuntimeError(f"Unsupported backbone without .layers: {type(backbone).__name__}")

    input_device = model.get_input_embeddings().weight.device
    output_device = next(output_head.parameters()).device
    print(f"dataset_rows={len(dataset)} layers={len(decoder_layers)} input_device={input_device}")


def release_similarity_resources() -> None:
    global dataset, model, tokenizer, backbone, output_head
    global decoder_layers, input_device, output_device

    dataset = None
    model = None
    tokenizer = None
    backbone = None
    output_head = None
    decoder_layers = None
    input_device = None
    output_device = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Prompt Filtering

A record is retained only when the base model predicts the expected first answer token for all four required roles. When several candidates are valid, a seeded deterministic selection avoids dataset-order and confidence-selection bias. Eligibility is model-specific, so cross-model comparisons should use the saved `source_index` values to restrict results to shared eligible records.

In [ ]:
ROLE_ORDER = ("edit_true", "paraphrase_true", "locality_true", "locality_new")
ROLE_TO_PROMPT_ID = {role: f"p{index}" for index, role in enumerate(ROLE_ORDER, start=1)}
PAIR_SPECS = (
    ("p1-p2", 0, 1, "edit - paraphrase"),
    ("p1-p3", 0, 2, "edit - locality true"),
    ("p1-p4", 0, 3, "edit - locality new"),
    ("p2-p3", 1, 2, "paraphrase - locality true"),
    ("p2-p4", 1, 3, "paraphrase - locality new"),
    ("p3-p4", 2, 3, "locality true - locality new"),
)


def answer_continuation(prompt: str, answer: str) -> str:
    if not answer:
        raise ValueError(f"Empty expected answer for prompt: {prompt!r}")
    if prompt and prompt[-1].isspace():
        return answer.lstrip()
    return answer if answer[0].isspace() else f" {answer}"


def expected_first_token(prompt: str, answer: str) -> tuple[int, str]:
    token_ids = tokenizer.encode(
        answer_continuation(prompt, answer),
        add_special_tokens=False,
    )
    if not token_ids:
        raise ValueError(f"Expected answer produced no tokens: {answer!r}")
    token_id = int(token_ids[0])
    return token_id, tokenizer.decode([token_id])


def build_filter_candidates(
    dataset,
    record_limit: int | None,
) -> tuple[list[dict], dict[int, dict]]:
    source_count = len(dataset) if record_limit is None else min(int(record_limit), len(dataset))
    candidates: list[dict] = []
    contexts: dict[int, dict] = {}

    for source_index in range(source_count):
        record = dataset[source_index]
        edit = record["edit"]
        true_answer = edit["answer"]["true"]
        new_answer = edit["answer"]["new"]
        contexts[source_index] = {
            "source_index": source_index,
            "subject": edit["subject"],
            "answers": {"true": true_answer, "new": new_answer},
        }

        role_candidates = [
            ("edit_true", 0, edit["prompt"], true_answer),
            *(
                ("paraphrase_true", index, prompt, true_answer)
                for index, prompt in enumerate(record["generality"])
            ),
        ]

        locality_group_counts = Counter()
        for index, item in enumerate(record["locality"]):
            if item["answer"] == true_answer:
                role = "locality_true"
            elif item["answer"] == new_answer:
                role = "locality_new"
            else:
                raise ValueError(
                    f"Record {source_index} has locality answer outside true/new groups: "
                    f"{item['answer']!r}"
                )
            locality_group_counts[role] += 1
            role_candidates.append((role, index, item["prompt"], item["answer"]))

        if locality_group_counts != Counter({"locality_true": 2, "locality_new": 2}):
            raise ValueError(
                f"Record {source_index} has unexpected locality split: "
                f"{dict(locality_group_counts)}"
            )

        for role, candidate_index, prompt, answer in role_candidates:
            expected_token_id, expected_token = expected_first_token(prompt, answer)
            candidates.append({
                "source_index": source_index,
                "role": role,
                "candidate_index": candidate_index,
                "prompt": prompt,
                "answer": answer,
                "expected_token_id": expected_token_id,
                "expected_token": expected_token,
            })

    return candidates, contexts


def predict_next_tokens(prompts: list[str]) -> tuple[list[int], list[float]]:
    batch = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
    ).to(input_device)
    with torch.inference_mode():
        outputs = backbone(**batch, use_cache=False, return_dict=True)
        final_hidden = outputs.last_hidden_state[:, -1, :].to(output_device)
        logits = output_head(final_hidden).float()
        top_logits, top_ids = logits.max(dim=-1)
        top_probabilities = (top_logits - torch.logsumexp(logits, dim=-1)).exp()

    result_ids = top_ids.cpu().tolist()
    result_probabilities = top_probabilities.cpu().tolist()
    del batch, outputs, final_hidden, logits, top_logits, top_ids, top_probabilities
    return result_ids, result_probabilities


def select_eligible_records(
    dataset,
    record_limit: int | None,
    filter_batch_size: int,
    selection_seed: int,
) -> tuple[list[dict], dict]:
    if filter_batch_size <= 0:
        raise ValueError("filter_batch_size must be positive.")
    candidates, contexts = build_filter_candidates(dataset, record_limit=record_limit)
    for start in tqdm(range(0, len(candidates), filter_batch_size), desc="Filter prompts"):
        batch_candidates = candidates[start : start + filter_batch_size]
        predicted_ids, predicted_probabilities = predict_next_tokens(
            [candidate["prompt"] for candidate in batch_candidates]
        )
        for candidate, predicted_id, probability in zip(
            batch_candidates, predicted_ids, predicted_probabilities
        ):
            candidate["predicted_token_id"] = int(predicted_id)
            candidate["predicted_token"] = tokenizer.decode([predicted_id])
            candidate["predicted_probability"] = float(probability)
            candidate["is_valid"] = predicted_id == candidate["expected_token_id"]

    candidates_by_record: dict[int, dict[str, list[dict]]] = defaultdict(lambda: defaultdict(list))
    total_candidate_counts = Counter()
    valid_candidate_counts = Counter()
    for candidate in candidates:
        candidates_by_record[candidate["source_index"]][candidate["role"]].append(candidate)
        total_candidate_counts[candidate["role"]] += 1
        if candidate["is_valid"]:
            valid_candidate_counts[candidate["role"]] += 1

    selected_records: list[dict] = []
    skip_reason_counts = Counter()
    for source_index, context in contexts.items():
        valid_by_role = {
            role: [candidate for candidate in candidates_by_record[source_index][role] if candidate["is_valid"]]
            for role in ROLE_ORDER
        }
        missing_roles = [role for role in ROLE_ORDER if not valid_by_role[role]]
        if missing_roles:
            skip_reason_counts.update(missing_roles)
            continue

        selected_prompts = []
        for role in ROLE_ORDER:
            chooser = Random(f"{selection_seed}:{source_index}:{role}")
            selected = dict(chooser.choice(valid_by_role[role]))
            selected.pop("is_valid", None)
            selected["id"] = ROLE_TO_PROMPT_ID[role]
            selected_prompts.append(selected)

        selected_records.append({**context, "prompts": selected_prompts})

    report = {
        "source_records": len(contexts),
        "eligible_records": len(selected_records),
        "skipped_records": len(contexts) - len(selected_records),
        "skip_reason_counts": dict(skip_reason_counts),
        "total_candidate_counts": dict(total_candidate_counts),
        "valid_candidate_counts": dict(valid_candidate_counts),
    }
    return selected_records, report

In [ ]:
def display_selection_report(selection_report: dict) -> None:
    display(pd.DataFrame([selection_report]))
    display(pd.DataFrame({
        "total": selection_report["total_candidate_counts"],
        "valid": selection_report["valid_candidate_counts"],
    }).rename_axis("role"))

## Layer Similarity Analysis

The embedding output is excluded. Layer `k` is the final prompt-token hidden state after transformer block `k`.

In [ ]:
def layer_structure() -> list[dict]:
    structures = []
    for index, layer in enumerate(decoder_layers):
        child_modules = {name: type(module).__name__ for name, module in layer.named_children()}
        structures.append({
            "index": index,
            "class": type(layer).__name__,
            "children": child_modules,
        })
    return structures


def model_metadata() -> dict:
    first_parameter = next(model.parameters())
    return {
        "model_key": MODEL_KEY,
        "checkpoint": model_source,
        "model_class": type(model).__name__,
        "backbone_class": type(backbone).__name__,
        "model_type": getattr(model.config, "model_type", None),
        "checkpoint_revision": getattr(model.config, "_commit_hash", None),
        "tokenizer_class": type(tokenizer).__name__,
        "num_hidden_layers": int(model.config.num_hidden_layers),
        "hidden_size": int(model.config.hidden_size),
        "num_parameters": sum(parameter.numel() for parameter in model.parameters()),
        "dtype": str(first_parameter.dtype).removeprefix("torch."),
        "representation": (
            "final non-padding prompt-token hidden state after each transformer block; "
            "embedding output excluded"
        ),
        "layers": layer_structure(),
    }


def analyze_record_batch(record_batch: list[dict]) -> list[dict]:
    prompts = [
        prompt_info["prompt"]
        for record in record_batch
        for prompt_info in record["prompts"]
    ]
    batch = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
    ).to(input_device)

    with torch.inference_mode():
        outputs = backbone(
            **batch,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )

    n_layers = len(decoder_layers)
    activations = torch.stack([
        hidden_state[:, -1, :].float().cpu()
        for hidden_state in outputs.hidden_states[1 : n_layers + 1]
    ])
    activations = activations.reshape(n_layers, len(record_batch), len(ROLE_ORDER), -1)
    normalized = F.normalize(activations, p=2, dim=-1)
    similarities = normalized @ normalized.transpose(-1, -2)

    analyzed_records = []
    for record_offset, record in enumerate(record_batch):
        pair_similarities = {
            pair_key: [
                round(float(value), 8)
                for value in similarities[:, record_offset, left_index, right_index].tolist()
            ]
            for pair_key, left_index, right_index, _ in PAIR_SPECS
        }
        analyzed_records.append({**record, "pair_similarities": pair_similarities})

    del batch, outputs, activations, normalized, similarities
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return analyzed_records


def aggregate_records(records: list[dict]) -> dict:
    if not records:
        raise ValueError("No eligible records were analyzed.")
    pairs = {}
    for pair_key, _, _, label in PAIR_SPECS:
        values = np.asarray(
            [record["pair_similarities"][pair_key] for record in records],
            dtype=np.float64,
        )
        pairs[pair_key] = {
            "label": label,
            "mean": np.round(values.mean(axis=0), 8).tolist(),
            "std": np.round(values.std(axis=0, ddof=0), 8).tolist(),
        }
    return {"record_count": len(records), "pairs": pairs}


def write_json_atomic(payload: dict, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(f"{destination.suffix}.tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=True, allow_nan=False)
    temporary.replace(destination)


def run_similarity_analysis(
    selected_records: list[dict],
    selection_report: dict,
    output_path: Path,
    similarity_record_batch_size: int,
    overwrite_output: bool,
    selection_seed: int,
) -> dict:
    if similarity_record_batch_size <= 0:
        raise ValueError("similarity_record_batch_size must be positive.")
    if output_path.exists() and not overwrite_output:
        raise FileExistsError(
            f"Output already exists: {output_path}. Enable overwrite to replace it."
        )

    completed_records: list[dict] = []
    for start in tqdm(
        range(0, len(selected_records), similarity_record_batch_size),
        desc="Analyze representations",
    ):
        batch_records = selected_records[start : start + similarity_record_batch_size]
        completed_records.extend(analyze_record_batch(batch_records))

    completed_records.sort(key=lambda record: record["source_index"])
    result = {
        "schema_version": 1,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "model": model_metadata(),
        "dataset": {
            "repo": DATASET_REPO,
            "config": DATASET_CONFIG,
            "split": DATASET_SPLIT,
            "rows": len(dataset),
            "fingerprint": getattr(dataset, "_fingerprint", None),
        },
        "selection": {
            "seed": selection_seed,
            "criterion": (
                "top-1 next-token ID equals the first token ID of the role's expected answer"
            ),
            **selection_report,
        },
        "prompt_roles": [
            {"id": ROLE_TO_PROMPT_ID[role], "role": role}
            for role in ROLE_ORDER
        ],
        "pair_order": [
            {"pair": pair_key, "label": label}
            for pair_key, _, _, label in PAIR_SPECS
        ],
        "records": completed_records,
        "aggregate": aggregate_records(completed_records),
    }
    write_json_atomic(result, output_path)
    return result

In [ ]:
def calculate_similarity(
    record_limit: int | None,
    filter_batch_size: int,
    similarity_record_batch_size: int,
    selection_seed: int,
    overwrite_output: bool,
) -> dict:
    output_path = similarity_output_path(record_limit)
    if output_path.exists() and not overwrite_output:
        raise FileExistsError(
            f"Output already exists: {output_path}. Enable overwrite to replace it."
        )

    try:
        load_similarity_resources()
        selected_records, selection_report = select_eligible_records(
            dataset,
            record_limit=record_limit,
            filter_batch_size=filter_batch_size,
            selection_seed=selection_seed,
        )
        display_selection_report(selection_report)
        result = run_similarity_analysis(
            selected_records,
            selection_report=selection_report,
            output_path=output_path,
            similarity_record_batch_size=similarity_record_batch_size,
            overwrite_output=overwrite_output,
            selection_seed=selection_seed,
        )
        print(f"saved={output_path} records={result['aggregate']['record_count']}")
        return result
    finally:
        release_similarity_resources()

## Inspect A Record And Plot Aggregate Results

In [ ]:
def prompt_table(record: dict) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "ID": prompt["id"],
            "Role": prompt["role"],
            "Prompt": prompt["prompt"],
            "Answer": prompt["answer"],
            "Expected token": repr(prompt["expected_token"]),
            "Predicted token": repr(prompt["predicted_token"]),
            "Top-1 probability": prompt["predicted_probability"],
            "Candidate index": prompt["candidate_index"],
        }
        for prompt in record["prompts"]
    ])


def similarity_table(record: dict) -> pd.DataFrame:
    n_layers = len(next(iter(record["pair_similarities"].values())))
    return pd.DataFrame.from_dict(
        record["pair_similarities"],
        orient="index",
        columns=[f"layer {index}" for index in range(n_layers)],
    ).rename_axis("Pair")


PAIR_COLORS = {
    "p1-p2": "#0072B2",
    "p1-p3": "#E69F00",
    "p1-p4": "#009E73",
    "p2-p3": "#D55E00",
    "p2-p4": "#56B4E9",
    "p3-p4": "#CC79A7",
}


def resolve_plot_layers(result: dict, layer_range: tuple[int, int] | None) -> np.ndarray:
    n_layers = int(result["model"]["num_hidden_layers"])
    if layer_range is None:
        return np.arange(n_layers)
    if len(layer_range) != 2:
        raise ValueError("layer_range must be None or a (start, end) tuple.")
    start_layer, end_layer = (int(value) for value in layer_range)
    if not 0 <= start_layer <= end_layer < n_layers:
        raise ValueError(
            f"Invalid inclusive layer range {layer_range}; model has layers 0..{n_layers - 1}."
        )
    return np.arange(start_layer, end_layer + 1)


def resolve_plot_pairs(pairs: list[str] | None) -> list[tuple]:
    pair_specs_by_key = {spec[0]: spec for spec in PAIR_SPECS}
    if pairs is None:
        return list(PAIR_SPECS)
    requested_pairs = list(dict.fromkeys(pairs))
    if not requested_pairs:
        raise ValueError("pairs must contain at least one pair when set.")
    unknown_pairs = [pair_key for pair_key in requested_pairs if pair_key not in pair_specs_by_key]
    if unknown_pairs:
        raise ValueError(f"Unknown prompt pairs: {unknown_pairs}")
    return [pair_specs_by_key[pair_key] for pair_key in requested_pairs]


def plot_similarity_aggregate(
    result: dict,
    layer_range: tuple[int, int] | None,
    pairs: list[str] | None,
    show_std: bool,
):
    aggregate_pairs = result["aggregate"]["pairs"]
    layers = resolve_plot_layers(result, layer_range=layer_range)
    pair_specs = resolve_plot_pairs(pairs)
    figure, axis = plt.subplots(figsize=(14, 7))

    for pair_key, _, _, label in pair_specs:
        mean_values = np.asarray(aggregate_pairs[pair_key]["mean"], dtype=float)[layers]
        color = PAIR_COLORS[pair_key]
        if show_std:
            std_values = np.asarray(aggregate_pairs[pair_key]["std"], dtype=float)[layers]
            lower = np.clip(mean_values - std_values, -1.0, 1.0)
            upper = np.clip(mean_values + std_values, -1.0, 1.0)
            axis.fill_between(layers, lower, upper, color=color, alpha=0.08, linewidth=0)
        axis.plot(layers, mean_values, color=color, linewidth=2.1, label=f"{pair_key}: {label}")

    axis.set_ylim(-1.0, 1.0)
    if len(layers) == 1:
        axis.set_xlim(layers[0] - 0.5, layers[0] + 0.5)
    else:
        axis.set_xlim(layers[0], layers[-1])
    tick_step = max(1, int(np.ceil(len(layers) / 16)))
    tick_layers = layers[::tick_step]
    if tick_layers[-1] != layers[-1]:
        tick_layers = np.append(tick_layers, layers[-1])
    axis.set_xticks(tick_layers)
    axis.set_xlabel("transformer layer")
    axis.set_ylabel("cosine similarity")
    axis.set_title(
        f"Representation similarity | {result['model']['model_key']} | "
        f"n={result['aggregate']['record_count']}",
        fontweight="bold",
    )
    axis.grid(alpha=0.22)
    axis.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), title="prompt pair")
    figure.tight_layout(rect=(0, 0, 0.78, 1))
    return figure, axis


def display_similarity_record(result: dict, position: int = 0) -> None:
    record = result["records"][position]
    print(f"position={position} source_index={record['source_index']} subject={record['subject']}")
    display(
        prompt_table(record)
        .style.hide(axis="index")
        .format({"Top-1 probability": "{:.4f}"})
    )
    display(similarity_table(record).style.format(precision=4))


def visualize_similarity(
    record_limit: int | None,
    record_position: int,
    layer_range: tuple[int, int] | None,
    pairs: list[str] | None,
    show_std: bool,
) -> dict:
    output_path = similarity_output_path(record_limit)
    if not output_path.exists():
        raise FileNotFoundError(
            f"Similarity result does not exist: {output_path}. Calculate it first."
        )
    with open(output_path, encoding="utf-8") as file:
        result = json.load(file)
    display_similarity_record(result, position=record_position)
    plot_similarity_aggregate(
        result,
        layer_range=layer_range,
        pairs=pairs,
        show_std=show_std,
    )
    return result

## Behavior Configuration

These options are consumed only by the final manual-action cell.

In [ ]:
SELECTION_SEED = 0
RECORD_LIMIT = None  # Set a positive integer for a diagnostic result file.
FILTER_BATCH_SIZE = 64
SIMILARITY_RECORD_BATCH_SIZE = 8  # Four prompts are analyzed per record.
OVERWRITE_EXISTING_RESULT = False

VISUALIZE_RECORD_POSITION = 0
PLOT_LAYER_RANGE = None  # None or an inclusive (start_layer, end_layer) tuple.
PLOT_PAIRS = None  # None or a list such as ["p1-p2", "p1-p3"].
PLOT_SHOW_STD = True

## Manual Actions

Uncomment one action at a time. Visualization reads only the existing JSON and never loads the model or dataset.

In [ ]:
# Calculate and save similarity JSON:
# similarity_result = calculate_similarity(
#     record_limit=RECORD_LIMIT,
#     filter_batch_size=FILTER_BATCH_SIZE,
#     similarity_record_batch_size=SIMILARITY_RECORD_BATCH_SIZE,
#     selection_seed=SELECTION_SEED,
#     overwrite_output=OVERWRITE_EXISTING_RESULT,
# )

# Visualize an existing similarity JSON:
# similarity_result = visualize_similarity(
#     record_limit=RECORD_LIMIT,
#     record_position=VISUALIZE_RECORD_POSITION,
#     layer_range=PLOT_LAYER_RANGE,
#     pairs=PLOT_PAIRS,
#     show_std=PLOT_SHOW_STD,
# )